# 원본 데이터 전처리작업

In [560]:
!pip3 install pyreadstat
!pip3 install scikit-learn
!pip3 install openai
!pip3 install transformers
!pip3 install tqdm
!pip3 install matplotlib


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip3 install --upgrade pip


In [561]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import pyreadstat

from tqdm import tqdm
from openai import OpenAI
from transformers import pipeline
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

In [562]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [563]:
#df1 is general, sample2 is multiculture
df1, meta = pyreadstat.read_sav('./general.sav')
df2, meta = pyreadstat.read_sav('./multiculture.sav')

In [564]:
df1.head()

,ID,SQ1,SQ2,SQ3,A1_1,A1_2,A1_3,A1_4,A1_5,A1_6,A1_7,A1_8,A1_9,A1_10,A1_11,A1_12,A1_13,A1_14,A1_15,A1_16,A1_17,A1_18,A1_19,A1_20,A1_21,A1_22,A1_23,A2_1,A2_2,A2_3,B2_1,B2_2,B2_3,B2_4,B2_5,B2_6,B2_7,B2_8,B2_9,B2_10,B2_11,B2_12,B2_13,B2_14,B2_15,B3,B4,B5,B6,B7,B8_1,B8_2,B8_3,B9_1,B9_2,B9_3,B9_4,B9_5,B9_6,B9_7,B9_8,B9_9,B9_10,B10,B11,B12_1,B12_2,B12_3,B12_4,B13,B14,B14_1,B14_2,B14_3,B14_4,B14_5,B14_6,B15_1,B15_2,B15_3,B15_4,B15_5,B15_6,B15_7,B16,C1_1,C1_2,C1_3,C2_1,C2_2,C3,C4,C5_1,C5_2,C5_3,C5_4,C5_5,C5_6,D1_1,D1_2,D1_3,D1_4,D1_5,D1_6,D1_7,D1_8,D1_9,D2,D3,D4,D5,D6,D7_1,D7_2,D7_3,D7_1_1,D8,D9_1,D9_2,D9_3,D10,E1_1,E1_2,E1_3,E1_4,E1_5,E1_6,E1_7,E1_8,E1_9,E1_10,E1_11,E1_12,E1_13,E1_14,E1_15,E1_16,E1_17,E1_18,E1_19,E2,E3,E4,E5,E6,E7_1,E7_2,F2,F3,F4,F5,F6,F7_1,F7_2,F8,F9,F10,F11,F12,DQ1,DQ2,DQ3A,DQ3B,DQ4
0,1005.0,1.0,13.0,2.0,5.0,4.0,4.0,1.0,3.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,1.0,5.0,5.0,1.0,1.0,1.0,1.0,1.0,4.0,5.0,5.0,4.0,5.0,5.0,5.0,5.0,5.0,2.0,1.0,1.0,5.0,5.0,5.0,5.0,5.0,5.0,1.0,1.0,1.0,1.0,5.0,2.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,4.0,2.0,5.0,5.0,5.0,5.0,2.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,4.0,5.0,5.0,2.0,2.0,2.0,1.0,3.0,2.0,0.0,0.0,0.0,0.0,4.0,2.0,3.0,2.0,3.0,2.0,4.0,5.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,1.0,1.0,1.0,2.0,3.0,2.0,4.0,3.0,2.0,6.0,6.0,2.0,1502.0,650.0,9.0,NaN
1,1006.0,1.0,11.0,1.0,5.0,5.0,2.0,4.0,3.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,3.0,1.0,4.0,1.0,5.0,5.0,1.0,2.0,1.0,1.0,1.0,2.0,5.0,5.0,3.0,4.0,3.0,1.0,2.0,1.0,5.0,4.0,1.0,5.0,4.0,1.0,1.0,5.0,5.0,2.0,1.0,2.0,1.0,4.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,4.0,2.0,4.0,5.0,5.0,3.0,1.0,2.0,1.0,99.0,99.0,99.0,99.0,99.0,4.0,3.0,4.0,3.0,3.0,1.0,1.0,4.0,0.0,0.0,0.0,3.0,0.0,1.0,4.0,2.0,4.0,1.0,1.0,1.0,2.0,4.0,1.0,1.0,1.0,1.0,1.0,2.0,5.0,1.0,1.0,1.0,2.0,2.0,7.0,1.0,99.0,99.0,2.0,1.0,5.0,99.0,99.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,2.0,2.0,1.0,3.0,2.0,1.0,4.0,4.0,2.0,1.0,5.0,2.0,1502.0,650.0,9.0,NaN
2,1007.0,2.0,12.0,1.0,5.0,5.0,5.0,1.0,3.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,5.0,5.0,3.0,3.0,3.0,3.0,1.0,5.0,5.0,5.0,4.0,5.0,4.0,5.0,3.0,5.0,5.0,5.0,5.0,4.0,5.0,3.0,5.0,5.0,5.0,2.0,1.0,1.0,1.0,5.0,3.0,3.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,4.0,1.0,5.0,5.0,5.0,5.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,5.0,5.0,5.0,2.0,4.0,1.0,1.0,4.0,2.0,0.0,1.0,0.0,0.0,4.0,4.0,4.0,1.0,1.0,1.0,5.0,5.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,4.0,99.0,99.0,NaN,1.0,8.0,99.0,99.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,1.0,1.0,1.0,2.0,2.0,2.0,9.0,9.0,99.0,4.0,3.0,2.0,1502.0,650.0,9.0,NaN
3,1008.0,1.0,13.0,2.0,5.0,4.0,3.0,2.0,1.0,1.0,1.0,2.0,2.0,1.0,1.0,2.0,4.0,3.0,3.0,1.0,4.0,4.0,2.0,2.0,2.0,2.0,2.0,3.0,4.0,4.0,5.0,5.0,4.0,4.0,5.0,5.0,5.0,5.0,3.0,5.0,5.0,4.0,5.0,3.0,4.0,2.0,1.0,2.0,1.0,5.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,4.0,3.0,3.0,3.0,4.0,4.0,1.0,2.0,1.0,99.0,99.0,99.0,99.0,99.0,4.0,4.0,5.0,2.0,1.0,9.0,1.0,4.0,2.0,2.0,1.0,0.0,0.0,2.0,3.0,3.0,2.0,1.0,1.0,3.0,4.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,2.0,1.0,1.0,2.0,2.0,1.0,4.0,4.0,3.0,4.0,4.0,2.0,1502.0,650.0,9.0,NaN
4,1306.0,1.0,12.0,1.0,3.0,5.0,4.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,5.0,5.0,4.0,4.0,3.0,3.0,1.0,1.0,1.0,1.0,2.0,2.0,5.0,3.0,1.0,3.0,1.0,1.0,2.0,5.0,1.0,1.0,1.0,1.0,5.0,3.0,5.0,1.0,5.0,2.0,1.0,1.0,1.0,3.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,3.0,1.0,1.0,1.0,4.0,4.0,1.0,2.0,3.0,1.0,1.0,4.0,1.0,99.0,99.0,99.0,99.0,99.0,3.0,4.0,5.0,3.0,4.0,1.0,1.0,3.0,1.0,0.0,1.0,5.0,2.0,2.0,5.0,5.0,2.0,4.0,4.0,3.0,3.0,2.0,1.0,2.0,1.0,1.0,1.0,1.0,2.0,1.0,8.0,2.0,2.0,2.0,4.0,1.0,99.0,99.0,4.0,1.0,7.0,99.0,99.0,1.0,1.0,1.0,1.

# 원본 전체데이터 평균이랑 분산 구하기

In [565]:
df1 = df1[['ID', 'F7_1', 'F7_2', 'F8', 'F9', 'F10', 'F11', 'F12', 'DQ2',
           'B2_1', 'B2_2', 'B2_3', 'B2_4', 'B2_5', 'B2_6', 'B2_7', 'B2_8', 'B2_9', 'B7',
           'B2_10', 'B2_12', 'B2_14',
           'B2_11', 'B2_13', 'B2_15',
           'B8_1', 'B8_2', 'B8_3',
           'B11', 'B12_3', 'B12_4', 'B15_1', 'B15_2', 'B15_3',
           'C3', 'C4', 'D1_1', 'D1_2', 'D1_3', 'D1_4', 'D1_5', 'D1_6', 'D1_7', 'D1_8', 'D1_9',
           'A1_1', 'A1_2', 'A1_3', 'A1_4', 'A1_5', 'A1_6', 'A1_7', 'A1_8', 'A1_9', 'A1_4', 'A1_10', 'A1_11', 'A1_12', 'A1_13', 'A1_14', 'A1_15' 
           ]]

In [566]:
df2 = df2[['ID', 'F7_1', 'F7_2', 'F8', 'F9', 'F10', 'F11', 'F12', 'DQ2',
           'B2_1', 'B2_2', 'B2_3', 'B2_4', 'B2_5', 'B2_6', 'B2_7', 'B2_8', 'B2_9', 'B7',
           'B2_10', 'B2_12', 'B2_14',
           'B2_11', 'B2_13', 'B2_15',
           'B8_1', 'B8_2', 'B8_3',
           'B11', 'B12_3', 'B12_4', 'B15_1', 'B15_2', 'B15_3',
           'C3', 'C4', 'D1_1', 'D1_2', 'D1_3', 'D1_4', 'D1_5', 'D1_6', 'D1_7', 'D1_8', 'D1_9',
           'A1_1', 'A1_2', 'A1_3', 'A1_4', 'A1_5', 'A1_6', 'A1_7', 'A1_8', 'A1_9', 'A1_4', 'A1_10', 'A1_11', 'A1_12', 'A1_13', 'A1_14', 'A1_15',
           #다문화 변수
           'A3', 'B2_16', 'B10', 'B17' 
           ]]

### 더미변환

In [567]:
#더미 변수 변환 할 것들 변환
#행정 구역
df1['DQ2'] = np.where(df1['DQ2']//100 == 1, 1, np.where(df1['DQ2']//100 == 8, 2, 3))
df2['DQ2'] = np.where(df2['DQ2']//100 == 1, 1, np.where(df2['DQ2']//100 == 8, 2, 3))

#부부 내 가정폭력 여부
#없으면 0, 한번이라도 있으면 1
df1['B8_1'] = df1['B8_1'].replace(9, np.nan)
df1['B8_2'] = df1['B8_2'].replace(9, np.nan)
df1['B8_3'] = df1['B8_3'].replace(9, np.nan)
df1['B8_1'] = df1['B8_1'].apply(lambda x: 0 if x == 1 else 1)
df1['B8_2'] = df1['B8_2'].apply(lambda x: 0 if x == 1 else 1)
df1['B8_3'] = df1['B8_3'].apply(lambda x: 0 if x == 1 else 1)

df2['B8_1'] = df2['B8_1'].replace(9, np.nan)
df2['B8_2'] = df2['B8_2'].replace(9, np.nan)
df2['B8_3'] = df2['B8_3'].replace(9, np.nan)
df2['B8_1'] = df2['B8_1'].apply(lambda x: 0 if x == 1 else 1)
df2['B8_2'] = df2['B8_2'].apply(lambda x: 0 if x == 1 else 1)
df2['B8_3'] = df2['B8_3'].apply(lambda x: 0 if x == 1 else 1)

df1['D8_violence'] = df1[['B8_1', 'B8_2', 'B8_3']].sum(axis = 1)
df2['D8_violence'] = df2[['B8_1', 'B8_2', 'B8_3']].sum(axis = 1)
df1 = df1.drop(['B8_1', 'B8_2', 'B8_3'], axis=1)
df2 = df2.drop(['B8_1', 'B8_2', 'B8_3'], axis=1)

#학교 폭력 피해 여부
#없으면 0, 한번이라도 있으면 1
df1['D1_1'] = df1['D1_1'].replace(9, np.nan)
df1['D1_2'] = df1['D1_2'].replace(9, np.nan)
df1['D1_3'] = df1['D1_3'].replace(9, np.nan)
df1['D1_4'] = df1['D1_4'].replace(9, np.nan)
df1['D1_5'] = df1['D1_5'].replace(9, np.nan)
df1['D1_6'] = df1['D1_6'].replace(9, np.nan)
df1['D1_7'] = df1['D1_7'].replace(9, np.nan)
df1['D1_8'] = df1['D1_8'].replace(9, np.nan)
df1['D1_9'] = df1['D1_9'].replace(9, np.nan)
df1['D1_1'] = df1['D1_1'].apply(lambda x: 0 if x == 1 else 1)
df1['D1_2'] = df1['D1_2'].apply(lambda x: 0 if x == 1 else 1)
df1['D1_3'] = df1['D1_3'].apply(lambda x: 0 if x == 1 else 1)
df1['D1_4'] = df1['D1_4'].apply(lambda x: 0 if x == 1 else 1)
df1['D1_5'] = df1['D1_5'].apply(lambda x: 0 if x == 1 else 1)
df1['D1_6'] = df1['D1_6'].apply(lambda x: 0 if x == 1 else 1)
df1['D1_7'] = df1['D1_7'].apply(lambda x: 0 if x == 1 else 1)
df1['D1_8'] = df1['D1_8'].apply(lambda x: 0 if x == 1 else 1)
df1['D1_9'] = df1['D1_9'].apply(lambda x: 0 if x == 1 else 1)

df2['D1_1'] = df2['D1_1'].replace(9, np.nan)
df2['D1_2'] = df2['D1_2'].replace(9, np.nan)
df2['D1_3'] = df2['D1_3'].replace(9, np.nan)
df2['D1_4'] = df2['D1_4'].replace(9, np.nan)
df2['D1_5'] = df2['D1_5'].replace(9, np.nan)
df2['D1_6'] = df2['D1_6'].replace(9, np.nan)
df2['D1_7'] = df2['D1_7'].replace(9, np.nan)
df2['D1_8'] = df2['D1_8'].replace(9, np.nan)
df2['D1_9'] = df2['D1_9'].replace(9, np.nan)
df2['D1_1'] = df2['D1_1'].apply(lambda x: 0 if x == 1 else 1)
df2['D1_2'] = df2['D1_2'].apply(lambda x: 0 if x == 1 else 1)
df2['D1_3'] = df2['D1_3'].apply(lambda x: 0 if x == 1 else 1)
df2['D1_4'] = df2['D1_4'].apply(lambda x: 0 if x == 1 else 1)
df2['D1_5'] = df2['D1_5'].apply(lambda x: 0 if x == 1 else 1)
df2['D1_6'] = df2['D1_6'].apply(lambda x: 0 if x == 1 else 1)
df2['D1_7'] = df2['D1_7'].apply(lambda x: 0 if x == 1 else 1)
df2['D1_8'] = df2['D1_8'].apply(lambda x: 0 if x == 1 else 1)
df2['D1_9'] = df2['D1_9'].apply(lambda x: 0 if x == 1 else 1)

df1['D1_violence'] = df1[['D1_1', 'D1_2', 'D1_3', 'D1_4', 'D1_5', 'D1_6', 'D1_7', 'D1_8', 'D1_9']].sum(axis = 1)
df2['D1_violence'] = df2[['D1_1', 'D1_2', 'D1_3', 'D1_4', 'D1_5', 'D1_6', 'D1_7', 'D1_8', 'D1_9']].sum(axis = 1)
df1 = df1.drop(['D1_1', 'D1_2', 'D1_3', 'D1_4', 'D1_5', 'D1_6', 'D1_7', 'D1_8', 'D1_9'], axis=1)
df2 = df2.drop(['D1_1', 'D1_2', 'D1_3', 'D1_4', 'D1_5', 'D1_6', 'D1_7', 'D1_8', 'D1_9'], axis=1)

#다문화 가정 차별 경험 유무
#없으면 0, 한번이라도 있으면 1
df2['B17'] = df2['B17'].replace(9, np.nan)
df2['B17'] = df2['B17'].apply(lambda x: 0 if x == 1 else 1)

In [568]:
df1['F12']= df1['F12'].replace(9, np.nan)

### 전처리 하기

In [569]:
df1['F7_1']= df1['F7_1'].replace(9, np.nan)
df1['F7_2'] = df1['F7_2'].replace(9, np.nan)
df1['F8'] = df1['F8'].replace(9, np.nan)
df1['F9']= df1['F9'].replace(9, np.nan)
df1['F10'] = df1['F10'].replace(99, np.nan)
df1['F11']= df1['F11'].replace(99, np.nan)
df1['F12']= df1['F12'].replace(9, np.nan)
df1['B11']= df1['B11'].replace(9, np.nan)
df1['C3']= df1['C3'].replace(9, np.nan)
df1['C4']= df1['C4'].replace(9, np.nan)

df2['F7_1']= df2['F7_1'].replace(9, np.nan)
df2['F7_2'] = df2['F7_2'].replace(9, np.nan)
df2['F8'] = df2['F8'].replace(9, np.nan)
df2['F9']= df2['F9'].replace(9, np.nan)
df2['F10'] = df2['F10'].replace(99, np.nan)
df2['F11']= df2['F11'].replace(99, np.nan)
df2['F12']= df2['F12'].replace(9, np.nan)
df2['B11']= df2['B11'].replace(9, np.nan)
df2['C3']= df2['C3'].replace(9, np.nan)
df2['C4']= df2['C4'].replace(9, np.nan)

In [570]:
# 결측치 최빈값으로 대체
cols = [
    'B2_1','B2_2','B2_3','B2_4','B2_5','B2_6','B2_7','B2_8','B2_9',
    'B2_10','B2_11','B2_12','B2_13','B2_14','B2_15',
    'B12_3','B12_4',
    'B15_1','B15_2','B15_3'
]

dfs = [df1, df2]

for df in dfs:
    for col in cols:
        m = df.loc[df[col] != 9, col].mode()[0]
        df[col] = df[col].mask(df[col] == 9, m)

# 가족 친밀도
# 1이 안친함 ~ 6이 친함
df1['B2_family'] = df1[['B2_1','B2_2','B2_3','B2_4','B2_5','B2_6','B2_7','B2_8','B2_9']].mean(axis=1)
df2['B2_family'] = df2[['B2_1','B2_2','B2_3','B2_4','B2_5','B2_6','B2_7','B2_8','B2_9']].mean(axis=1)
df1 = df1.drop(['B2_1','B2_2','B2_3','B2_4','B2_5','B2_6','B2_7','B2_8','B2_9'], axis=1)
df2 = df2.drop(['B2_1','B2_2','B2_3','B2_4','B2_5','B2_6','B2_7','B2_8','B2_9'], axis=1)

# 아버지와의 친밀도
# 1이 안친함 ~ 6이 친함
df1['B2_father'] = df1[['B2_10','B2_12','B2_14']].mean(axis=1)
df2['B2_father'] = df2[['B2_10','B2_12','B2_14']].mean(axis=1)
df1 = df1.drop(['B2_10','B2_12','B2_14'], axis = 1)
df2 = df2.drop(['B2_10','B2_12','B2_14'], axis = 1)

# 어머니와의 친밀도
# 1이 안친함 ~ 6이 친함
df1['B2_mother'] = df1[['B2_11','B2_13','B2_15']].mean(axis=1)
df2['B2_mother'] = df2[['B2_11','B2_13','B2_15']].mean(axis=1)
df1 = df1.drop(['B2_11','B2_13','B2_15'], axis = 1)
df2 = df2.drop(['B2_11','B2_13','B2_15'], axis = 1)

# 선생님과의 지지도
# 1이 안지지 ~ 6이 지지
df1['B12_teacher'] = df1[['B12_3','B12_4']].mean(axis=1)
df2['B12_teacher'] = df2[['B12_3','B12_4']].mean(axis=1)
df1 = df1.drop(['B12_3','B12_4'], axis = 1)
df2 = df2.drop(['B12_3','B12_4'], axis = 1)

# 친구들과의 지지도
# 1이 안지지 ~ 6이 지지
df1['B15_friend'] = df1[['B15_1','B15_2','B15_3']].mean(axis=1)
df2['B15_friend'] = df2[['B15_1','B15_2','B15_3']].mean(axis=1)
df1 = df1.drop(['B15_1','B15_2','B15_3'], axis = 1)
df2 = df2.drop(['B15_1','B15_2','B15_3'], axis = 1)

In [571]:
#우울도
#1(자기긍정) ~ 5(자기부정) - 높을수록 우울감이 높음
# A1_1: 나는 나 자신에 대해 대체로 만족한다
df1['A1_1'] = 6 - df1['A1_1']
df2['A1_1'] = 6 - df2['A1_1']

# A1_2: 나는 다른 사람만큼 중요한 사람이다
df1['A1_2'] = 6 - df1['A1_2']
df2['A1_2'] = 6 - df2['A1_2']

# A1_3: 나는 장점이 많다고 생각한다
df1['A1_3'] = 6 - df1['A1_3']
df2['A1_3'] = 6 - df2['A1_3']

df1['A1_dpression'] = df1[['A1_1', 'A1_2', 'A1_3', 'A1_4', 'A1_5', 'A1_6', 'A1_7', 'A1_8', 'A1_9', 'A1_10', 'A1_11', 'A1_12', 'A1_13', 'A1_14', 'A1_15']].sum(axis = 1)
df2['A1_dpression'] = df2[['A1_1', 'A1_2', 'A1_3', 'A1_4', 'A1_5', 'A1_6', 'A1_7', 'A1_8', 'A1_9', 'A1_10', 'A1_11', 'A1_12', 'A1_13', 'A1_14', 'A1_15']].sum(axis = 1)
df1 = df1.drop(['A1_1', 'A1_2', 'A1_3', 'A1_4', 'A1_5', 'A1_6', 'A1_7', 'A1_8', 'A1_9', 'A1_10', 'A1_11', 'A1_12', 'A1_13', 'A1_14', 'A1_15'], axis=1)
df2 = df2.drop(['A1_1', 'A1_2', 'A1_3', 'A1_4', 'A1_5', 'A1_6', 'A1_7', 'A1_8', 'A1_9', 'A1_10', 'A1_11', 'A1_12', 'A1_13', 'A1_14', 'A1_15'], axis=1)

In [572]:
df1.describe().round(2)

,ID,F7_1,F7_2,F8,F9,F10,F11,F12,DQ2,B7,B11,C3,C4,D8_violence,D1_violence,B2_family,B2_father,B2_mother,B12_teacher,B15_friend,A1_dpression
count,800.00,751.00,757.00,776.00,777.00,787.00,792.00,796.00,800.00,800.00,797.00,797.00,799.00,800.00,800.00,800.00,800.00,800.00,800.00,800.00,800.00
mean,91434.80,2.09,1.84,3.72,3.54,2.96,4.03,4.36,2.16,4.31,2.58,2.07,2.75,0.92,1.13,3.84,3.56,4.08,3.83,3.98,32.94
std,57794.04,0.47,0.51,0.91,0.80,1.47,1.88,0.99,0.74,0.98,0.96,1.14,1.57,1.16,1.86,0.80,1.09,0.89,0.96,0.83,10.87
min,1005.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.00,0.00,1.22,1.00,1.00,1.00,1.00,16.00
25%,50980.50,2.00,2.00,3.00,3.00,2.00,2.00,4.00,2.00,4.00,2.00,1.00,1.00,0.00,0.00,3.33,3.00,3.67,3.00,3.33,25.00
50%,84620.50,2.00,2.00,4.00,3.00,2.00,4.00,4.00,2.00,4.00,2.00,2.00,2.00,0.00,0.00,3.89,3.67,4.33,4.00,4.00,31.00
75%,137024.25,2.00,2.00,4.00,4.00,4.00,6.00,5.00,3.00,5.00,3.00,3.00,4.00,2.00,2.00,4.47,4.33,5.00,4.50,4.67,41.00
max,197406.00,3.00,3.00,6.00,6.00,8.00,8.00,7.00,3.00,9.00,5.00,6.00,7.00,3.00,9.00,5.00,5.00,5.00,5.00,5.00,80.00


In [573]:
df2.describe().round(2)

,ID,F7_1,F7_2,F8,F9,F10,F11,F12,DQ2,B7,B11,C3,C4,A3,B2_16,B10,B17,D8_violence,D1_violence,B2_family,B2_father,B2_mother,B12_teacher,B15_friend,A1_dpression
count,800.00,703.00,727.00,763.00,756.00,782.00,785.00,797.00,800.00,800.00,799.00,796.00,797.00,800.00,800.00,800.00,800.00,800.00,800.00,800.00,800.00,800.00,800.00,800.00,800.00
mean,138625.38,2.36,1.83,3.27,3.49,3.65,4.17,3.98,2.42,4.06,3.07,2.38,3.00,1.97,4.25,3.79,0.28,1.23,1.23,3.49,3.24,3.71,3.69,3.79,34.64
std,70477.28,0.55,0.61,1.15,0.88,1.62,1.79,0.99,0.74,1.16,1.01,1.28,1.73,4.96,1.05,0.46,0.45,1.26,2.09,0.80,1.08,0.96,0.94,0.92,10.98
min,1001.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.00,0.00,0.00,1.00,1.00,1.00,1.00,1.00,16.00
25%,82201.75,2.00,1.00,3.00,3.00,2.00,2.00,3.00,2.00,3.00,2.00,1.00,2.00,1.00,4.00,4.00,0.00,0.00,0.00,3.00,2.67,3.00,3.00,3.00,26.00
50%,154801.50,2.00,2.00,3.00,3.00,3.00,4.00,4.00,3.00,4.00,3.00,2.00,3.00,1.00,5.00,4.00,0.00,1.00,0.00,3.44,3.33,3.67,4.00,4.00,34.00
75%,206202.25,3.00,2.00,4.00,4.00,4.00,6.00,4.00,3.00,5.00,4.00,3.00,4.00,3.00,5.00,4.00,1.00,3.00,2.00,4.11,4.00,4.33,4.50,4.33,42.00
max,221303.00,3.00,3.00,6.00,6.00,8.00,8.00,7.00,3.00,9.00,5.00,6.00,7.00,99.00,9.00,4.00,1.00,3.00,9.00,5.00,5.00,5.00,5.00,5.00,72.00


# 추출한 샘플만 비교하기

In [574]:
#sample1 is general, sample2 is multiculture
sample1 = pd.read_csv('./sample_general.csv')
sample2 = pd.read_csv('./sample_multi.csv')

In [575]:
# 역맵핑
sample1['SQ1'] = sample1['SQ1'].map({'남성': 1, '여성': 2})
sample2['SQ1'] = sample2['SQ1'].map({'남성': 1, '여성': 2})

reverse_mapping = {
    '한국': 0, '중국': 1, '일본': 3, '태국': 6, '몽골': 7, '러시아': 8, '우즈베키스탄': 12,
    '미국': 14, '인도': 15, '인도네시아': 16, '파키스탄': 17, '스페인': 18, '방글라데시': 19,
    '에콰도르': 20, '호주': 22, '사우디아라비아': 24, '캐나다': 26, '대만': 29, '페루': 35, '모르는 곳': 99
}
sample2['F1_1'] = sample2['F1_1'].map(reverse_mapping)

def dq2_reverse(x):
    if x == '수도인 서울':
        return 100
    elif x == '수도권':
        return 800
    else:
        return 200
sample1['DQ2'] = sample1['DQ2'].apply(dq2_reverse)
sample2['DQ2'] = sample2['DQ2'].apply(dq2_reverse)

reverse_mapping = {
    '초등학교 5학년': 1, '초등학교 6학년': 2, '중학교 1학년': 3, '중학교 2학년': 4,
    '중학교 3학년': 5, '고등학교 1학년': 6, '고등학교 2학년': 7, '고등학교 3학년': 8
}
sample1['SQ3'] = sample1['SQ3'].map(reverse_mapping)
sample2['SQ3'] = sample2['SQ3'].map(reverse_mapping)

reverse_mapping = {
    '중국 (조선족)': 1, '중국 (한족)': 2, '일본': 3, '베트남': 4, '필리핀': 5, '태국': 6,
    '몽골': 7, '러시아': 8, '기타': 9, '한국': 10
}
sample2['B1_2'] = sample2['B1_2'].map(reverse_mapping)
sample2['B1_1'] = sample2['B1_1'].map(reverse_mapping)

reverse_mapping = {'결혼': 1, '이혼': 2, '사별': 3, '재혼': 4, '별거': 5}
sample1['F5'] = sample1['F5'].map(reverse_mapping)
sample2['F5'] = sample2['F5'].map(reverse_mapping)

reverse_mapping = {
    '부모님 없이': 1, '친부모 두 분과': 2, '오직 아버지와': 3, '오직 어머니와': 4,
    '아버지와 새어머니와': 5, '어머니와 새아버지와': 6, '부모님이 아닌 다른 분들과': 8
}
sample1['F6'] = sample1['F6'].map(reverse_mapping)
sample2['F6'] = sample2['F6'].map(reverse_mapping)

reverse_mapping = {
    '제일 못 사는': 1,
    '어느정도 못 사는': 2,
    '약간 못 사는': 3,
    '중간 정도 사는': 4,
    '약간 잘 사는': 5,
    '어느정도 잘 사는': 6,
    '제일 잘 사는': 7
}
sample1['F12'] = sample1['F12'].map(reverse_mapping)
sample2['F12'] = sample2['F12'].map(reverse_mapping)

In [576]:
sample1 = sample1[['ID', 'F7_1', 'F7_2', 'F8', 'F9', 'F10', 'F11', 'F12', 'DQ2',
           'B2_1', 'B2_2', 'B2_3', 'B2_4', 'B2_5', 'B2_6', 'B2_7', 'B2_8', 'B2_9', 'B7',
           'B2_10', 'B2_12', 'B2_14',
           'B2_11', 'B2_13', 'B2_15',
           'B8_1', 'B8_2', 'B8_3',
           'B11', 'B12_3', 'B12_4', 'B15_1', 'B15_2', 'B15_3',
           'C3', 'C4', 'D1_1', 'D1_2', 'D1_3', 'D1_4', 'D1_5', 'D1_6', 'D1_7', 'D1_8', 'D1_9',
           'A1_1', 'A1_2', 'A1_3', 'A1_4', 'A1_5', 'A1_6', 'A1_7', 'A1_8', 'A1_9', 'A1_4', 'A1_10', 'A1_11', 'A1_12', 'A1_13', 'A1_14', 'A1_15' 
           ]]

In [577]:
sample2 = sample2[['ID', 'F7_1', 'F7_2', 'F8', 'F9', 'F10', 'F11', 'F12', 'DQ2',
           'B2_1', 'B2_2', 'B2_3', 'B2_4', 'B2_5', 'B2_6', 'B2_7', 'B2_8', 'B2_9', 'B7',
           'B2_10', 'B2_12', 'B2_14',
           'B2_11', 'B2_13', 'B2_15',
           'B8_1', 'B8_2', 'B8_3',
           'B11', 'B12_3', 'B12_4', 'B15_1', 'B15_2', 'B15_3',
           'C3', 'C4', 'D1_1', 'D1_2', 'D1_3', 'D1_4', 'D1_5', 'D1_6', 'D1_7', 'D1_8', 'D1_9',
           'A1_1', 'A1_2', 'A1_3', 'A1_4', 'A1_5', 'A1_6', 'A1_7', 'A1_8', 'A1_9', 'A1_4', 'A1_10', 'A1_11', 'A1_12', 'A1_13', 'A1_14', 'A1_15',
           #다문화 변수
           'A3', 'B2_16', 'B10', 'B17' 
           ]]

In [578]:
#더미 변수 변환 할 것들 변환
#행정 구역
sample1['DQ2'] = np.where(sample1['DQ2']//100 == 1, 1, np.where(sample1['DQ2']//100 == 8, 2, 3))
sample2['DQ2'] = np.where(sample2['DQ2']//100 == 1, 1, np.where(sample2['DQ2']//100 == 8, 2, 3))

#부부 내 가정폭력 여부
#없으면 0, 한번이라도 있으면 1
sample1['B8_1'] = sample1['B8_1'].replace(9, np.nan)
sample1['B8_2'] = sample1['B8_2'].replace(9, np.nan)
sample1['B8_3'] = sample1['B8_3'].replace(9, np.nan)
sample1['B8_1'] = sample1['B8_1'].apply(lambda x: 0 if x == 1 else 1)
sample1['B8_2'] = sample1['B8_2'].apply(lambda x: 0 if x == 1 else 1)
sample1['B8_3'] = sample1['B8_3'].apply(lambda x: 0 if x == 1 else 1)

sample2['B8_1'] = sample2['B8_1'].replace(9, np.nan)
sample2['B8_2'] = sample2['B8_2'].replace(9, np.nan)
sample2['B8_3'] = sample2['B8_3'].replace(9, np.nan)
sample2['B8_1'] = sample2['B8_1'].apply(lambda x: 0 if x == 1 else 1)
sample2['B8_2'] = sample2['B8_2'].apply(lambda x: 0 if x == 1 else 1)
sample2['B8_3'] = sample2['B8_3'].apply(lambda x: 0 if x == 1 else 1)

sample1['D8_violence'] = sample1[['B8_1', 'B8_2', 'B8_3']].sum(axis = 1)
sample2['D8_violence'] = sample2[['B8_1', 'B8_2', 'B8_3']].sum(axis = 1)
sample1 = sample1.drop(['B8_1', 'B8_2', 'B8_3'], axis=1)
sample2 = sample2.drop(['B8_1', 'B8_2', 'B8_3'], axis=1)

#학교 폭력 피해 여부
#없으면 0, 한번이라도 있으면 1
sample1['D1_1'] = sample1['D1_1'].replace(9, np.nan)
sample1['D1_2'] = sample1['D1_2'].replace(9, np.nan)
sample1['D1_3'] = sample1['D1_3'].replace(9, np.nan)
sample1['D1_4'] = sample1['D1_4'].replace(9, np.nan)
sample1['D1_5'] = sample1['D1_5'].replace(9, np.nan)
sample1['D1_6'] = sample1['D1_6'].replace(9, np.nan)
sample1['D1_7'] = sample1['D1_7'].replace(9, np.nan)
sample1['D1_8'] = sample1['D1_8'].replace(9, np.nan)
sample1['D1_9'] = sample1['D1_9'].replace(9, np.nan)
sample1['D1_1'] = sample1['D1_1'].apply(lambda x: 0 if x == 1 else 1)
sample1['D1_2'] = sample1['D1_2'].apply(lambda x: 0 if x == 1 else 1)
sample1['D1_3'] = sample1['D1_3'].apply(lambda x: 0 if x == 1 else 1)
sample1['D1_4'] = sample1['D1_4'].apply(lambda x: 0 if x == 1 else 1)
sample1['D1_5'] = sample1['D1_5'].apply(lambda x: 0 if x == 1 else 1)
sample1['D1_6'] = sample1['D1_6'].apply(lambda x: 0 if x == 1 else 1)
sample1['D1_7'] = sample1['D1_7'].apply(lambda x: 0 if x == 1 else 1)
sample1['D1_8'] = sample1['D1_8'].apply(lambda x: 0 if x == 1 else 1)
sample1['D1_9'] = sample1['D1_9'].apply(lambda x: 0 if x == 1 else 1)

sample2['D1_1'] = sample2['D1_1'].replace(9, np.nan)
sample2['D1_2'] = sample2['D1_2'].replace(9, np.nan)
sample2['D1_3'] = sample2['D1_3'].replace(9, np.nan)
sample2['D1_4'] = sample2['D1_4'].replace(9, np.nan)
sample2['D1_5'] = sample2['D1_5'].replace(9, np.nan)
sample2['D1_6'] = sample2['D1_6'].replace(9, np.nan)
sample2['D1_7'] = sample2['D1_7'].replace(9, np.nan)
sample2['D1_8'] = sample2['D1_8'].replace(9, np.nan)
sample2['D1_9'] = sample2['D1_9'].replace(9, np.nan)
sample2['D1_1'] = sample2['D1_1'].apply(lambda x: 0 if x == 1 else 1)
sample2['D1_2'] = sample2['D1_2'].apply(lambda x: 0 if x == 1 else 1)
sample2['D1_3'] = sample2['D1_3'].apply(lambda x: 0 if x == 1 else 1)
sample2['D1_4'] = sample2['D1_4'].apply(lambda x: 0 if x == 1 else 1)
sample2['D1_5'] = sample2['D1_5'].apply(lambda x: 0 if x == 1 else 1)
sample2['D1_6'] = sample2['D1_6'].apply(lambda x: 0 if x == 1 else 1)
sample2['D1_7'] = sample2['D1_7'].apply(lambda x: 0 if x == 1 else 1)
sample2['D1_8'] = sample2['D1_8'].apply(lambda x: 0 if x == 1 else 1)
sample2['D1_9'] = sample2['D1_9'].apply(lambda x: 0 if x == 1 else 1)

sample1['D1_violence'] = sample1[['D1_1', 'D1_2', 'D1_3', 'D1_4', 'D1_5', 'D1_6', 'D1_7', 'D1_8', 'D1_9']].sum(axis = 1)
sample2['D1_violence'] = sample2[['D1_1', 'D1_2', 'D1_3', 'D1_4', 'D1_5', 'D1_6', 'D1_7', 'D1_8', 'D1_9']].sum(axis = 1)
sample1 = sample1.drop(['D1_1', 'D1_2', 'D1_3', 'D1_4', 'D1_5', 'D1_6', 'D1_7', 'D1_8', 'D1_9'], axis=1)
sample2 = sample2.drop(['D1_1', 'D1_2', 'D1_3', 'D1_4', 'D1_5', 'D1_6', 'D1_7', 'D1_8', 'D1_9'], axis=1)

#다문화 가정 차별 경험 유무
#없으면 0, 한번이라도 있으면 1
sample2['B17'] = sample2['B17'].replace(9, np.nan)
sample2['B17'] = sample2['B17'].apply(lambda x: 0 if x == 1 else 1)

In [579]:
sample1['F7_1']= sample1['F7_1'].replace(9, np.nan)
sample1['F7_2'] = sample1['F7_2'].replace(9, np.nan)
sample1['F8'] = sample1['F8'].replace(9, np.nan)
sample1['F9']= sample1['F9'].replace(9, np.nan)
sample1['F10'] = sample1['F10'].replace(99, np.nan)
sample1['F11']= sample1['F11'].replace(99, np.nan)
sample1['F12']= sample1['F12'].replace(9, np.nan)
sample1['B11']= sample1['B11'].replace(9, np.nan)
sample1['C3']= sample1['C3'].replace(9, np.nan)
sample1['C4']= sample1['C4'].replace(9, np.nan)

sample2['F7_1']= sample2['F7_1'].replace(9, np.nan)
sample2['F7_2'] = sample2['F7_2'].replace(9, np.nan)
sample2['F8'] = sample2['F8'].replace(9, np.nan)
sample2['F9']= sample2['F9'].replace(9, np.nan)
sample2['F10'] = sample2['F10'].replace(99, np.nan)
sample2['F11']= sample2['F11'].replace(99, np.nan)
sample2['F12']= sample2['F12'].replace(9, np.nan)
sample2['B11']= sample2['B11'].replace(9, np.nan)
sample2['C3']= sample2['C3'].replace(9, np.nan)
sample2['C4']= sample2['C4'].replace(9, np.nan)

In [580]:
# 결측치 최빈값으로 대체
cols = [
    'B2_1','B2_2','B2_3','B2_4','B2_5','B2_6','B2_7','B2_8','B2_9',
    'B2_10','B2_11','B2_12','B2_13','B2_14','B2_15',
    'B12_3','B12_4',
    'B15_1','B15_2','B15_3'
]

samples = [sample1, sample2]

for sample in samples:
    for col in cols:
        m = sample.loc[sample[col] != 9, col].mode()[0]
        sample[col] = sample[col].mask(sample[col] == 9, m)

# 가족 친밀도
# 1이 안친함 ~ 6이 친함
sample1['B2_family'] = sample1[['B2_1','B2_2','B2_3','B2_4','B2_5','B2_6','B2_7','B2_8','B2_9']].mean(axis=1)
sample2['B2_family'] = sample2[['B2_1','B2_2','B2_3','B2_4','B2_5','B2_6','B2_7','B2_8','B2_9']].mean(axis=1)
sample1 = sample1.drop(['B2_1','B2_2','B2_3','B2_4','B2_5','B2_6','B2_7','B2_8','B2_9'], axis=1)
sample2 = sample2.drop(['B2_1','B2_2','B2_3','B2_4','B2_5','B2_6','B2_7','B2_8','B2_9'], axis=1)

# 아버지와의 친밀도
# 1이 안친함 ~ 6이 친함
sample1['B2_father'] = sample1[['B2_10','B2_12','B2_14']].mean(axis=1)
sample2['B2_father'] = sample2[['B2_10','B2_12','B2_14']].mean(axis=1)
sample1 = sample1.drop(['B2_10','B2_12','B2_14'], axis = 1)
sample2 = sample2.drop(['B2_10','B2_12','B2_14'], axis = 1)

# 어머니와의 친밀도
# 1이 안친함 ~ 6이 친함
sample1['B2_mother'] = sample1[['B2_11','B2_13','B2_15']].mean(axis=1)
sample2['B2_mother'] = sample2[['B2_11','B2_13','B2_15']].mean(axis=1)
sample1 = sample1.drop(['B2_11','B2_13','B2_15'], axis = 1)
sample2 = sample2.drop(['B2_11','B2_13','B2_15'], axis = 1)

# 선생님과의 지지도
# 1이 안지지 ~ 6이 지지
sample1['B12_teacher'] = sample1[['B12_3','B12_4']].mean(axis=1)
sample2['B12_teacher'] = sample2[['B12_3','B12_4']].mean(axis=1)
sample1 = sample1.drop(['B12_3','B12_4'], axis = 1)
sample2 = sample2.drop(['B12_3','B12_4'], axis = 1)

# 친구들과의 지지도
# 1이 안지지 ~ 6이 지지
sample1['B15_friend'] = sample1[['B15_1','B15_2','B15_3']].mean(axis=1)
sample2['B15_friend'] = sample2[['B15_1','B15_2','B15_3']].mean(axis=1)
sample1 = sample1.drop(['B15_1','B15_2','B15_3'], axis = 1)
sample2 = sample2.drop(['B15_1','B15_2','B15_3'], axis = 1)

In [581]:
#우울도
#1(자기긍정) ~ 5(자기부정) - 높을수록 우울감이 높음
# A1_1: 나는 나 자신에 대해 대체로 만족한다
sample1['A1_1'] = 6 - sample1['A1_1']
sample2['A1_1'] = 6 - sample2['A1_1']

# A1_2: 나는 다른 사람만큼 중요한 사람이다
sample1['A1_2'] = 6 - sample1['A1_2']
sample2['A1_2'] = 6 - sample2['A1_2']

# A1_3: 나는 장점이 많다고 생각한다
sample1['A1_3'] = 6 - sample1['A1_3']
sample2['A1_3'] = 6 - sample2['A1_3']

sample1['A1_dpression'] = sample1[['A1_1', 'A1_2', 'A1_3', 'A1_4', 'A1_5', 'A1_6', 'A1_7', 'A1_8', 'A1_9', 'A1_10', 'A1_11', 'A1_12', 'A1_13', 'A1_14', 'A1_15']].sum(axis = 1)
sample2['A1_dpression'] = sample2[['A1_1', 'A1_2', 'A1_3', 'A1_4', 'A1_5', 'A1_6', 'A1_7', 'A1_8', 'A1_9', 'A1_10', 'A1_11', 'A1_12', 'A1_13', 'A1_14', 'A1_15']].sum(axis = 1)
sample1 = sample1.drop(['A1_1', 'A1_2', 'A1_3', 'A1_4', 'A1_5', 'A1_6', 'A1_7', 'A1_8', 'A1_9', 'A1_10', 'A1_11', 'A1_12', 'A1_13', 'A1_14', 'A1_15'], axis=1)
sample2 = sample2.drop(['A1_1', 'A1_2', 'A1_3', 'A1_4', 'A1_5', 'A1_6', 'A1_7', 'A1_8', 'A1_9', 'A1_10', 'A1_11', 'A1_12', 'A1_13', 'A1_14', 'A1_15'], axis=1)

In [582]:
sample1.describe().round(2)

,ID,F7_1,F7_2,F8,F9,F10,F11,F12,DQ2,B7,B11,C3,C4,D8_violence,D1_violence,B2_family,B2_father,B2_mother,B12_teacher,B15_friend,A1_dpression
count,50.00,48.00,49.00,47.00,47.00,50.00,50.0,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00
mean,99153.36,2.15,1.90,3.81,3.55,2.84,3.7,4.30,2.08,4.24,2.54,2.22,3.06,0.94,1.50,3.89,3.45,4.00,3.87,4.01,33.26
std,55383.69,0.46,0.47,0.92,0.77,1.43,1.8,1.16,0.75,0.94,0.97,1.43,1.50,1.22,2.03,0.66,1.18,0.97,0.86,0.77,12.07
min,3711.00,1.00,1.00,1.00,1.00,1.00,1.0,1.00,1.00,1.00,1.00,1.00,1.00,0.00,0.00,2.33,1.00,1.33,2.00,2.33,18.00
25%,54191.00,2.00,2.00,3.00,3.00,2.00,2.0,4.00,2.00,4.00,2.00,1.00,2.00,0.00,0.00,3.56,2.67,3.67,3.00,3.33,22.00
50%,100509.00,2.00,2.00,4.00,3.00,2.00,3.5,4.00,2.00,4.00,2.00,2.00,3.00,0.00,1.00,3.89,3.67,4.33,4.00,4.00,32.00
75%,138183.00,2.00,2.00,4.00,4.00,4.00,6.0,5.00,3.00,5.00,3.00,3.00,4.00,1.75,2.00,4.42,4.33,4.67,4.50,4.67,41.75
max,195617.00,3.00,3.00,6.00,5.00,8.00,6.0,7.00,3.00,6.00,5.00,6.00,7.00,3.00,8.00,5.00,5.00,5.00,5.00,5.00,60.00


In [583]:
sample2.describe().round(2)

,ID,F7_1,F7_2,F8,F9,F10,F11,F12,DQ2,B7,B11,C3,C4,A3,B2_16,B10,B17,D8_violence,D1_violence,B2_family,B2_father,B2_mother,B12_teacher,B15_friend,A1_dpression
count,50.00,47.00,44.00,47.00,48.00,49.00,49.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00
mean,148424.50,2.36,1.80,3.19,3.65,3.73,3.96,4.10,2.38,4.10,3.06,2.60,3.24,1.80,4.20,3.80,0.26,1.08,1.02,3.34,3.11,3.53,3.74,3.99,34.06
std,63830.46,0.57,0.63,1.08,0.98,1.67,1.99,0.89,0.75,1.11,1.06,1.29,1.72,1.03,1.39,0.45,0.44,1.24,1.74,0.89,1.15,0.98,0.97,0.92,11.52
min,6901.00,1.00,1.00,1.00,2.00,1.00,1.00,2.00,1.00,2.00,1.00,1.00,1.00,1.00,1.00,2.00,0.00,0.00,0.00,1.56,1.00,1.00,1.50,1.00,16.00
25%,102302.50,2.00,1.00,3.00,3.00,2.00,2.00,4.00,2.00,3.00,2.00,2.00,2.00,1.00,4.00,4.00,0.00,0.00,0.00,2.56,2.42,3.00,3.00,3.33,23.25
50%,156502.00,2.00,2.00,3.00,3.00,3.00,4.00,4.00,3.00,4.00,3.00,2.50,3.00,1.00,4.50,4.00,0.00,1.00,0.00,3.28,3.00,3.67,3.75,4.00,35.00
75%,207951.00,3.00,2.00,4.00,4.00,5.00,6.00,4.75,3.00,5.00,3.75,3.00,4.00,3.00,5.00,4.00,0.75,2.00,1.00,4.08,4.00,4.00,4.50,4.92,42.00
max,220601.00,3.00,3.00,6.00,6.00,8.00,8.00,7.00,3.00,6.00,5.00,6.00,7.00,4.00,9.00,4.00,1.00,3.00,7.00,5.00,5.00,5.00,5.00,5.00,58.00


# Synthetic Data 값

In [584]:
synthetic1 = pd.read_csv('./general_answer.csv')
synthetic1 = synthetic1.drop('Unnamed: 0', axis=1)
synthetic2 = pd.read_csv('./multiculture_answer.csv')
synthetic2 = synthetic2.drop('Unnamed: 0', axis=1)

In [585]:
def extract_answer(text):
    matches = list(re.finditer(r'\(([A-G])\)', text))
    return matches[-1].group(1) if matches else None

# answer 열 추가
synthetic1['final'] = synthetic1['initial_answer'].apply(extract_answer)
synthetic2['final'] = synthetic2['answer'].apply(extract_answer)

In [586]:
def making_answer(text):
    if '4' in text:
        return 'B'
    return None

synthetic1['final'] = synthetic1['initial_answer'].apply(extract_answer)
synthetic2['final'] = synthetic2['answer'].apply(extract_answer)

In [587]:
synthetic1.head()

,질문_ID,persona,question,model,initial_answer,final
0,53618.0,"지금은 2012년 7월이다. 너는 12.0살 여성이며, 대한민국의 수도권에 12.0년 째 살고 있는 초등학교 5학년 학생이다. 어머니와 아버지의 국적은 한국이다. 부모님은 현재 결혼 중이며 너는 친부모 두 분과 살고 있다. 가족의 소득은 중간 정도 사는 정도이다.",당신의 아버지의 나이는?,meta-llama/Llama-4-Scout-17B-16E-Instruct,(B) 40대,B
1,53618.0,"지금은 2012년 7월이다. 너는 12.0살 여성이며, 대한민국의 수도권에 12.0년 째 살고 있는 초등학교 5학년 학생이다. 어머니와 아버지의 국적은 한국이다. 부모님은 현재 결혼 중이며 너는 친부모 두 분과 살고 있다. 가족의 소득은 중간 정도 사는 정도이다.",당신의 아버지의 나이는?,google/gemma-3-12b-it,"음... 아빠는 회사에서 되게 씩씩하시고, 엄마보다 조금 더 늦게 주무시는데... (B) 40대 일 것 같아요!",B
2,134608.0,"지금은 2012년 7월이다. 너는 14.0살 남성이며, 대한민국의 지방에 14.0년 째 살고 있는 중학교 1학년 학생이다. 어머니와 아버지의 국적은 한국이다. 부모님은 현재 결혼 중이며 너는 친부모 두 분과 살고 있다. 가족의 소득은 약간 잘 사는 정도이다.",당신의 아버지의 나이는?,meta-llama/Llama-4-Scout-17B-16E-Instruct,"저는 중학교 1학년이고, 올해 14살입니다. 아버지는 40대이실 것 같아요. 제 어머니도 40대이실 것 같고요. 그러니까 제 아버지는 (B) 40대라고 생각합니다.",B
3,134608.0,"지금은 2012년 7월이다. 너는 14.0살 남성이며, 대한민국의 지방에 14.0년 째 살고 있는 중학교 1학년 학생이다. 어머니와 아버지의 국적은 한국이다. 부모님은 현재 결혼 중이며 너는 친부모 두 분과 살고 있다. 가족의 소득은 약간 잘 사는 정도이다.",당신의 아버지의 나이는?,google/gemma-3-12b-it,"(B) 40대\n\n아빠는… 회사 엄청 오래 다니시니까, 40대 중반쯤 되신 것 같아요. 잘나가시는 편이라 아빠도 젊어 보이시긴 하는데, 엄마가 가끔 “너 나이 먹는 거 안 보이나?” 그러시면 아빠가 툴툴대세요. ㅋㅋㅋ",B
4,60109.0,"지금은 2012년 7월이다. 너는 11.0살 남성이며, 대한민국의 수도인 서울에 11.0년 째 살고 있는 초등학교 5학년 학생이다. 어머니와 아버지의 국적은 한국이다. 부모님은 현재 재혼 중이며 너는 아버지와 새어머니와 살고 있다. 가족의 소득은 중간 정도 사는 정도이다.",당신의 아버지의 나이는?,meta-llama/Llama-4-Scout-17B-16E-Instruct,"저는 11살이라 잘 모르지만, 우리 아버지는 40살이 넘으셨다고 했던 것 같아요. 제 동생은 아직 없지만요. 그래서 (B) 40대 같습니다.",B


In [588]:
qf = pd.read_csv('./question.csv')

In [589]:
def replace_bracket_text(row):
    if pd.notnull(row['Specific']):
        return re.sub(r'\[.*?\]', str(row['Specific']), row['Question'])
    else:
        return row['Question']

qf['Question'] = qf.apply(replace_bracket_text, axis=1)
qf = qf.drop(columns=['Specific']) #Specific 열 삭제

In [590]:
max_options = 7

def extract_options(row):
    pattern = r'\(([a-g])\)([^()]+)'
    options = re.findall(pattern, row['Question'])
    question_only = re.split(r'\([a-h]\)', row['Question'])[0].strip()
    option_texts = [opt[1].strip() for opt in options]
    option_texts += [np.nan] * (max_options - len(option_texts))
    result = {'Question': question_only}
    for i, opt in enumerate(option_texts):
        result[f'option_{chr(ord("a")+i)}'] = opt
    return pd.Series(result)

# 적용
qf_new = qf.apply(extract_options, axis=1)
qf_new['Tag'] = qf['Tag']
qf_new['Number'] = qf['Number']

In [591]:
qf_new.head()

,Question,option_a,option_b,option_c,option_d,option_e,option_f,option_g,Tag,Number
0,당신의 아버지의 나이는?,30대 이하,40대,50대이상,NaN,NaN,NaN,NaN,Demographic,F7_1
1,당신의 어머니의 나이는?,30대 이하,40대,50대이상,NaN,NaN,NaN,NaN,Demographic,F7_2
2,당신의 아버지는 학교를 어디까지 다니셨나요?,초등학교 졸업 또는 중퇴,중학교 졸업 또는 중퇴,고등학교 졸업 또는 중퇴,대학교 졸업 또는 중퇴,대학원 이상,아버지 안계심,NaN,Demographic,F8
3,당신의 어머니는 학교를 어디까지 다니셨나요?,초등학교 졸업 또는 중퇴,중학교 졸업 또는 중퇴,고등학교 졸업 또는 중퇴,대학교 졸업 또는 중퇴,대학원 이상,어머니 안계심,NaN,Demographic,F9
4,당신의 아버지의 직업은 무엇인가요?,"전문직 예를 들어 의사, 교사 등","사무직 예를 들어 공무원, 회사원 등",공장 및 건설노동자,"판매/ 서비스직 예를 들어 상점이나 식당 경영, 혹은 종업원 등",농어업,직업이 없음,아버지 안계심,Demographic,F10


In [592]:
qf1 = qf_new[qf_new['Tag'] != 'Multiculture']
qf2 = qf_new

In [593]:
synthetic1 = synthetic1.merge(
    qf1[['Question', 'Number']],
    left_on='question',
    right_on='Question',
    how='left'
)

synthetic1 = synthetic1.drop(['Question'], axis=1)

In [594]:
synthetic2 = synthetic2.merge(
    qf2[['Question', 'Number']],
    left_on='question',
    right_on='Question',
    how='left'
)
synthetic2 = synthetic2.drop(['Question'], axis=1)

In [595]:
answer_dict = {'A': 1,
               'B': 2,
               'C': 3,
               'D': 4,
               'E': 5,
               'F': 6,
               'G': 7}

synthetic1['final'] = synthetic1['final'].map(answer_dict)
synthetic2['final'] = synthetic2['final'].map(answer_dict)

In [596]:
synthetic1_llama = synthetic1[synthetic1['model'] == 'meta-llama/Llama-4-Scout-17B-16E-Instruct']
synthetic2_llama = synthetic2[synthetic2['model'] == 'meta-llama/Llama-4-Scout-17B-16E-Instruct']
synthetic1_llama = synthetic1_llama.drop('model', axis=1)
synthetic2_llama = synthetic2_llama.drop('model', axis=1)

synthetic1_gemmni = synthetic1[synthetic1['model'] == 'google/gemma-3-12b-it']
synthetic2_gemmni = synthetic2[synthetic2['model'] == 'google/gemma-3-12b-it']
synthetic1_gemmni = synthetic1_gemmni.drop('model', axis=1)
synthetic2_gemmni = synthetic2_gemmni.drop('model', axis=1)

In [597]:
synthetic1_llama['final'] = synthetic1_llama['final'].fillna(2.0)
synthetic2_llama['final'] = synthetic2_llama['final'].fillna(2.0)

synthetic1_gemmni['final'] = synthetic1_gemmni['final'].fillna(2.0)
synthetic2_gemmni['final'] = synthetic2_gemmni['final'].fillna(2.0)

In [598]:
synthetic1_llama = synthetic1_llama[['질문_ID', 'final', 'Number']]
synthetic2_llama = synthetic2_llama[['질문_ID', 'final', 'Number']]
synthetic1_gemmni = synthetic1_gemmni[['질문_ID', 'final', 'Number']]
synthetic2_gemmni = synthetic2_gemmni[['질문_ID', 'final', 'Number']]

synthetic1_llama = synthetic1_llama.rename(columns={'질문_ID': 'ID'})
synthetic2_llama = synthetic2_llama.rename(columns={'질문_ID': 'ID'})
synthetic1_gemmni = synthetic1_gemmni.rename(columns={'질문_ID': 'ID'})
synthetic2_gemmni = synthetic2_gemmni.rename(columns={'질문_ID': 'ID'})

In [599]:
synthetic1_llama_p = synthetic1_llama.pivot(index='ID', columns='Number', values='final')
synthetic1_gemmni_p = synthetic1_gemmni.pivot(index='ID', columns='Number', values='final')

In [600]:
synthetic2_llama_p = synthetic2_llama.pivot(index='ID', columns='Number', values='final')
synthetic2_gemmni_p = synthetic2_gemmni.pivot(index='ID', columns='Number', values='final')

In [601]:
#부부 내 가정폭력 여부
#없으면 0, 한번이라도 있으면 1
synthetic1_llama_p['D8_violence'] = synthetic1_llama_p[['B8_1', 'B8_2', 'B8_3']].sum(axis = 1)
synthetic2_llama_p['D8_violence'] = synthetic2_llama_p[['B8_1', 'B8_2', 'B8_3']].sum(axis = 1)
synthetic1_llama_p = synthetic1_llama_p.drop(['B8_1', 'B8_2', 'B8_3'], axis=1)
synthetic2_llama_p = synthetic2_llama_p.drop(['B8_1', 'B8_2', 'B8_3'], axis=1)

#학교 폭력 피해 여부
#없으면 0, 한번이라도 있으면 1
synthetic1_llama_p['D1_violence'] = synthetic1_llama_p[['D1_1', 'D1_2', 'D1_3', 'D1_4', 'D1_5', 'D1_6', 'D1_7', 'D1_8', 'D1_9']].sum(axis = 1)
synthetic2_llama_p['D1_violence'] = synthetic2_llama_p[['D1_1', 'D1_2', 'D1_3', 'D1_4', 'D1_5', 'D1_6', 'D1_7', 'D1_8', 'D1_9']].sum(axis = 1)
synthetic1_llama_p = synthetic1_llama_p.drop(['D1_1', 'D1_2', 'D1_3', 'D1_4', 'D1_5', 'D1_6', 'D1_7', 'D1_8', 'D1_9'], axis=1)
synthetic2_llama_p = synthetic2_llama_p.drop(['D1_1', 'D1_2', 'D1_3', 'D1_4', 'D1_5', 'D1_6', 'D1_7', 'D1_8', 'D1_9'], axis=1)

In [602]:
# 가족 친밀도
# 1이 안친함 ~ 6이 친함
synthetic1_llama_p['B2_family'] = synthetic1_llama_p[['B2_1','B2_2','B2_3','B2_4','B2_6','B2_7','B2_8','B2_9']].mean(axis=1)
synthetic2_llama_p['B2_family'] = synthetic2_llama_p[['B2_1','B2_2','B2_3','B2_4','B2_6','B2_7','B2_8','B2_9']].mean(axis=1)
synthetic1_llama_p = synthetic1_llama_p.drop(['B2_1','B2_2','B2_3','B2_4','B2_6','B2_7','B2_8','B2_9'], axis=1)
synthetic2_llama_p = synthetic2_llama_p.drop(['B2_1','B2_2','B2_3','B2_4','B2_6','B2_7','B2_8','B2_9'], axis=1)

# 아버지와의 친밀도
# 1이 안친함 ~ 6이 친함
synthetic1_llama_p['B2_father'] = synthetic1_llama_p[['B2_10','B2_12','B2_14']].mean(axis=1)
synthetic2_llama_p['B2_father'] = synthetic2_llama_p[['B2_10','B2_12','B2_14']].mean(axis=1)
synthetic1_llama_p = synthetic1_llama_p.drop(['B2_10','B2_12','B2_14'], axis = 1)
synthetic2_llama_p = synthetic2_llama_p.drop(['B2_10','B2_12','B2_14'], axis = 1)

# 어머니와의 친밀도
# 1이 안친함 ~ 6이 친함
synthetic1_llama_p['B2_mother'] = synthetic1_llama_p[['B2_11','B2_13','B2_15']].mean(axis=1)
synthetic2_llama_p['B2_mother'] = synthetic2_llama_p[['B2_11','B2_13','B2_15']].mean(axis=1)
synthetic1_llama_p = synthetic1_llama_p.drop(['B2_11','B2_13','B2_15'], axis = 1)
synthetic2_llama_p = synthetic2_llama_p.drop(['B2_11','B2_13','B2_15'], axis = 1)

# 선생님과의 지지도
# 1이 안지지 ~ 6이 지지
synthetic1_llama_p['B12_teacher'] = synthetic1_llama_p[['B12_3','B12_4']].mean(axis=1)
synthetic2_llama_p['B12_teacher'] = synthetic2_llama_p[['B12_3','B12_4']].mean(axis=1)
synthetic1_llama_p = synthetic1_llama_p.drop(['B12_3','B12_4'], axis = 1)
synthetic2_llama_p = synthetic2_llama_p.drop(['B12_3','B12_4'], axis = 1)

# 친구들과의 지지도
# 1이 안지지 ~ 6이 지지
synthetic1_llama_p['B15_friend'] = synthetic1_llama_p[['B15_1','B15_2','B15_3']].mean(axis=1)
synthetic2_llama_p['B15_friend'] = synthetic2_llama_p[['B15_1','B15_2','B15_3']].mean(axis=1)
synthetic1_llama_p = synthetic1_llama_p.drop(['B15_1','B15_2','B15_3'], axis = 1)
synthetic2_llama_p = synthetic2_llama_p.drop(['B15_1','B15_2','B15_3'], axis = 1)

In [603]:
#우울도
#1(자기긍정) ~ 5(자기부정) - 높을수록 우울감이 높음
# A1_1: 나는 나 자신에 대해 대체로 만족한다
synthetic1_llama_p['A1_1'] = 6 - synthetic1_llama_p['A1_1']
synthetic2_llama_p['A1_1'] = 6 - synthetic2_llama_p['A1_1']

# A1_2: 나는 다른 사람만큼 중요한 사람이다
synthetic1_llama_p['A1_2'] = 6 - synthetic1_llama_p['A1_2']
synthetic2_llama_p['A1_2'] = 6 - synthetic2_llama_p['A1_2']

# A1_3: 나는 장점이 많다고 생각한다
synthetic1_llama_p['A1_3'] = 6 - synthetic1_llama_p['A1_3']
synthetic2_llama_p['A1_3'] = 6 - synthetic2_llama_p['A1_3']

synthetic1_llama_p['A1_dpression'] = synthetic1_llama_p[['A1_1', 'A1_2', 'A1_3', 'A1_4', 'A1_5', 'A1_6', 'A1_7', 'A1_8', 'A1_9', 'A1_10', 'A1_11', 'A1_12', 'A1_13', 'A1_14', 'A1_15']].sum(axis = 1)
synthetic2_llama_p['A1_dpression'] = synthetic2_llama_p[['A1_1', 'A1_2', 'A1_3', 'A1_4', 'A1_5', 'A1_6', 'A1_7', 'A1_8', 'A1_9', 'A1_10', 'A1_11', 'A1_12', 'A1_13', 'A1_14', 'A1_15']].sum(axis = 1)
synthetic1_llama_p = synthetic1_llama_p.drop(['A1_1', 'A1_2', 'A1_3', 'A1_4', 'A1_5', 'A1_6', 'A1_7', 'A1_8', 'A1_9', 'A1_10', 'A1_11', 'A1_12', 'A1_13', 'A1_14', 'A1_15'], axis=1)
synthetic2_llama_p = synthetic2_llama_p.drop(['A1_1', 'A1_2', 'A1_3', 'A1_4', 'A1_5', 'A1_6', 'A1_7', 'A1_8', 'A1_9', 'A1_10', 'A1_11', 'A1_12', 'A1_13', 'A1_14', 'A1_15'], axis=1)

In [604]:
#부부 내 가정폭력 여부
synthetic1_gemmni_p['D8_violence'] = synthetic1_gemmni_p[['B8_1', 'B8_2', 'B8_3']].sum(axis = 1)
synthetic2_gemmni_p['D8_violence'] = synthetic2_gemmni_p[['B8_1', 'B8_2', 'B8_3']].sum(axis = 1)
synthetic1_gemmni_p = synthetic1_gemmni_p.drop(['B8_1', 'B8_2', 'B8_3'], axis=1)
synthetic2_gemmni_p = synthetic2_gemmni_p.drop(['B8_1', 'B8_2', 'B8_3'], axis=1)

#학교 폭력 피해 여부
synthetic1_gemmni_p['D1_violence'] = synthetic1_gemmni_p[['D1_1', 'D1_2', 'D1_3', 'D1_4', 'D1_5', 'D1_6', 'D1_7', 'D1_8', 'D1_9']].sum(axis = 1)
synthetic2_gemmni_p['D1_violence'] = synthetic2_gemmni_p[['D1_1', 'D1_2', 'D1_3', 'D1_4', 'D1_5', 'D1_6', 'D1_7', 'D1_8', 'D1_9']].sum(axis = 1)
synthetic1_gemmni_p = synthetic1_gemmni_p.drop(['D1_1', 'D1_2', 'D1_3', 'D1_4', 'D1_5', 'D1_6', 'D1_7', 'D1_8', 'D1_9'], axis=1)
synthetic2_gemmni_p = synthetic2_gemmni_p.drop(['D1_1', 'D1_2', 'D1_3', 'D1_4', 'D1_5', 'D1_6', 'D1_7', 'D1_8', 'D1_9'], axis=1)

In [605]:
# 가족 친밀도
# 1이 안친함 ~ 6이 친함
synthetic1_gemmni_p['B2_family'] = synthetic1_gemmni_p[['B2_1','B2_2','B2_3','B2_4','B2_6','B2_7','B2_8','B2_9']].mean(axis=1)
synthetic2_gemmni_p['B2_family'] = synthetic2_gemmni_p[['B2_1','B2_2','B2_3','B2_4','B2_6','B2_7','B2_8','B2_9']].mean(axis=1)
synthetic1_gemmni_p = synthetic1_gemmni_p.drop(['B2_1','B2_2','B2_3','B2_4','B2_6','B2_7','B2_8','B2_9'], axis=1)
synthetic2_gemmni_p = synthetic2_gemmni_p.drop(['B2_1','B2_2','B2_3','B2_4','B2_6','B2_7','B2_8','B2_9'], axis=1)

# 아버지와의 친밀도
# 1이 안친함 ~ 6이 친함
synthetic1_gemmni_p['B2_father'] = synthetic1_gemmni_p[['B2_10','B2_12','B2_14']].mean(axis=1)
synthetic2_gemmni_p['B2_father'] = synthetic2_gemmni_p[['B2_10','B2_12','B2_14']].mean(axis=1)
synthetic1_gemmni_p = synthetic1_gemmni_p.drop(['B2_10','B2_12','B2_14'], axis = 1)
synthetic2_gemmni_p = synthetic2_gemmni_p.drop(['B2_10','B2_12','B2_14'], axis = 1)

# 어머니와의 친밀도
# 1이 안친함 ~ 6이 친함
synthetic1_gemmni_p['B2_mother'] = synthetic1_gemmni_p[['B2_11','B2_13','B2_15']].mean(axis=1)
synthetic2_gemmni_p['B2_mother'] = synthetic2_gemmni_p[['B2_11','B2_13','B2_15']].mean(axis=1)
synthetic1_gemmni_p = synthetic1_gemmni_p.drop(['B2_11','B2_13','B2_15'], axis = 1)
synthetic2_gemmni_p = synthetic2_gemmni_p.drop(['B2_11','B2_13','B2_15'], axis = 1)

# 선생님과의 지지도
# 1이 안지지 ~ 6이 지지
synthetic1_gemmni_p['B12_teacher'] = synthetic1_gemmni_p[['B12_3','B12_4']].mean(axis=1)
synthetic2_gemmni_p['B12_teacher'] = synthetic2_gemmni_p[['B12_3','B12_4']].mean(axis=1)
synthetic1_gemmni_p = synthetic1_gemmni_p.drop(['B12_3','B12_4'], axis = 1)
synthetic2_gemmni_p = synthetic2_gemmni_p.drop(['B12_3','B12_4'], axis = 1)

# 친구들과의 지지도
# 1이 안지지 ~ 6이 지지
synthetic1_gemmni_p['B15_friend'] = synthetic1_gemmni_p[['B15_1','B15_2','B15_3']].mean(axis=1)
synthetic2_gemmni_p['B15_friend'] = synthetic2_gemmni_p[['B15_1','B15_2','B15_3']].mean(axis=1)
synthetic1_gemmni_p = synthetic1_gemmni_p.drop(['B15_1','B15_2','B15_3'], axis = 1)
synthetic2_gemmni_p = synthetic2_gemmni_p.drop(['B15_1','B15_2','B15_3'], axis = 1)

In [606]:
#우울도
#1(자기긍정) ~ 5(자기부정) - 높을수록 우울감이 높음
# A1_1: 나는 나 자신에 대해 대체로 만족한다
synthetic1_gemmni_p['A1_1'] = 6 - synthetic1_gemmni_p['A1_1']
synthetic2_gemmni_p['A1_1'] = 6 - synthetic2_gemmni_p['A1_1']

# A1_2: 나는 다른 사람만큼 중요한 사람이다
synthetic1_gemmni_p['A1_2'] = 6 - synthetic1_gemmni_p['A1_2']
synthetic2_gemmni_p['A1_2'] = 6 - synthetic2_gemmni_p['A1_2']

# A1_3: 나는 장점이 많다고 생각한다
synthetic1_gemmni_p['A1_3'] = 6 - synthetic1_gemmni_p['A1_3']
synthetic2_gemmni_p['A1_3'] = 6 - synthetic2_gemmni_p['A1_3']

synthetic1_gemmni_p['A1_dpression'] = synthetic1_gemmni_p[['A1_1', 'A1_2', 'A1_3', 'A1_4', 'A1_5', 'A1_6', 'A1_7', 'A1_8', 'A1_9', 'A1_10', 'A1_11', 'A1_12', 'A1_13', 'A1_14', 'A1_15']].sum(axis = 1)
synthetic2_gemmni_p['A1_dpression'] = synthetic2_gemmni_p[['A1_1', 'A1_2', 'A1_3', 'A1_4', 'A1_5', 'A1_6', 'A1_7', 'A1_8', 'A1_9', 'A1_10', 'A1_11', 'A1_12', 'A1_13', 'A1_14', 'A1_15']].sum(axis = 1)
synthetic1_gemmni_p = synthetic1_gemmni_p.drop(['A1_1', 'A1_2', 'A1_3', 'A1_4', 'A1_5', 'A1_6', 'A1_7', 'A1_8', 'A1_9', 'A1_10', 'A1_11', 'A1_12', 'A1_13', 'A1_14', 'A1_15'], axis=1)
synthetic2_gemmni_p = synthetic2_gemmni_p.drop(['A1_1', 'A1_2', 'A1_3', 'A1_4', 'A1_5', 'A1_6', 'A1_7', 'A1_8', 'A1_9', 'A1_10', 'A1_11', 'A1_12', 'A1_13', 'A1_14', 'A1_15'], axis=1)

In [607]:
synthetic1_llama_p.describe().round(2)

Number,B11,B7,C3,C4,DQ2,F10,F11,F7_1,F7_2,F8,F9,D8_violence,D1_violence,B2_family,B2_father,B2_mother,B12_teacher,B15_friend,A1_dpression
count,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00
mean,2.54,3.92,3.16,3.66,2.48,2.60,3.20,2.02,1.94,3.22,2.94,5.84,11.48,3.96,3.80,3.97,3.98,3.99,47.42
std,0.76,0.53,0.37,0.48,1.22,1.54,1.54,0.14,0.24,0.74,0.31,0.47,1.05,0.19,0.61,0.19,0.10,0.09,3.12
min,1.00,2.00,3.00,3.00,1.00,1.00,1.00,2.00,1.00,3.00,2.00,4.00,10.00,3.12,2.00,2.67,3.50,3.33,44.00
25%,2.00,4.00,3.00,3.00,2.00,2.00,1.00,2.00,2.00,3.00,3.00,6.00,11.00,4.00,4.00,4.00,4.00,4.00,45.00
50%,3.00,4.00,3.00,4.00,2.00,2.00,4.00,2.00,2.00,3.00,3.00,6.00,12.00,4.00,4.00,4.00,4.00,4.00,47.00
75%,3.00,4.00,3.00,4.00,4.00,2.00,4.00,2.00,2.00,3.00,3.00,6.00,12.00,4.00,4.00,4.00,4.00,4.00,49.00
max,4.00,5.00,4.00,4.00,4.00,7.00,7.00,3.00,2.00,6.00,4.00,6.00,16.00,4.12,4.00,4.00,4.00,4.00,58.00


In [608]:
synthetic2_llama_p.describe().round(2)

Number,A3,B10,B11,B17,B2_16,B7,C3,C4,DQ2,F10,F11,F7_1,F7_2,F8,F9,D8_violence,D1_violence,B2_family,B2_father,B2_mother,B12_teacher,B15_friend,A1_dpression
count,50.0,50.00,50.00,50.00,50.0,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.0,50.00
mean,3.0,3.02,3.02,1.04,4.0,3.92,3.08,3.56,3.04,2.44,4.24,1.92,1.38,3.10,3.20,5.52,10.28,3.92,3.90,3.86,3.99,4.0,50.26
std,0.0,0.14,0.74,0.20,0.0,0.70,0.27,0.50,1.23,1.03,1.02,0.27,0.49,0.51,0.88,0.74,0.67,0.24,0.35,0.39,0.07,0.0,2.55
min,3.0,3.00,1.00,1.00,4.0,2.00,3.00,3.00,1.00,2.00,1.00,1.00,1.00,2.00,2.00,3.00,10.00,3.12,2.67,2.67,3.50,4.0,45.00
25%,3.0,3.00,3.00,1.00,4.0,4.00,3.00,3.00,2.00,2.00,4.00,2.00,1.00,3.00,3.00,5.00,10.00,4.00,4.00,4.00,4.00,4.0,48.25
50%,3.0,3.00,3.00,1.00,4.0,4.00,3.00,4.00,4.00,2.00,4.00,2.00,1.00,3.00,3.00,6.00,10.00,4.00,4.00,4.00,4.00,4.0,50.00
75%,3.0,3.00,3.00,1.00,4.0,4.00,3.00,4.00,4.00,3.00,4.00,2.00,2.00,3.00,4.00,6.00,10.00,4.00,4.00,4.00,4.00,4.0,52.00
max,3.0,4.00,5.00,2.00,4.0,6.00,4.00,4.00,4.00,7.00,7.00,2.00,2.00,6.00,6.00,6.00,14.00,4.12,4.00,4.00,4.00,4.0,54.00


In [609]:
synthetic1_gemmni_p.describe().round(2)

Number,B11,B7,C3,C4,DQ2,F10,F11,F7_1,F7_2,F8,F9,D8_violence,D1_violence,B2_family,B2_father,B2_mother,B12_teacher,B15_friend,A1_dpression
count,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00
mean,2.64,3.64,3.20,4.12,2.40,2.42,2.44,2.02,1.92,3.50,3.44,5.36,10.14,3.17,2.54,2.80,2.92,3.01,43.38
std,0.75,0.85,0.64,0.63,1.18,1.14,1.03,0.25,0.40,0.74,0.61,0.56,1.37,0.58,0.38,0.33,0.31,0.29,4.57
min,1.00,1.00,2.00,3.00,1.00,2.00,1.00,1.00,1.00,2.00,1.00,4.00,9.00,1.38,1.00,1.67,1.50,2.33,34.00
25%,2.00,3.00,3.00,4.00,2.00,2.00,2.00,2.00,2.00,3.00,3.00,5.00,9.00,2.84,2.67,2.67,3.00,2.67,41.25
50%,3.00,4.00,3.00,4.00,2.00,2.00,2.00,2.00,2.00,3.00,3.00,5.00,9.00,3.25,2.67,2.67,3.00,3.00,45.00
75%,3.00,4.00,4.00,4.00,4.00,2.00,2.00,2.00,2.00,4.00,4.00,6.00,12.00,3.50,2.67,3.00,3.00,3.33,45.75
max,4.00,5.00,5.00,6.00,4.00,7.00,7.00,3.00,3.00,6.00,4.00,6.00,12.00,4.25,2.67,3.33,3.50,3.33,58.00


In [610]:
synthetic2_gemmni_p.describe().round(2)

Number,A3,B10,B11,B17,B2_16,B7,C3,C4,DQ2,F10,F11,F7_1,F7_2,F8,F9,D8_violence,D1_violence,B2_family,B2_father,B2_mother,B12_teacher,B15_friend,A1_dpression
count,50.0,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.0,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00
mean,3.0,3.02,2.90,1.30,3.32,3.80,3.04,3.86,2.92,2.26,3.3,2.10,1.98,3.54,2.78,5.14,10.10,3.01,2.68,2.79,2.89,2.84,44.40
std,0.0,0.14,0.54,0.46,0.51,1.14,0.53,0.50,1.23,0.78,1.3,0.36,0.14,0.86,1.74,0.61,0.86,0.57,0.16,0.32,0.23,0.25,3.92
min,3.0,3.00,1.00,1.00,2.00,1.00,2.00,2.00,1.00,2.00,1.0,1.00,1.00,3.00,1.00,3.00,9.00,1.50,2.00,2.00,2.00,2.33,34.00
25%,3.0,3.00,3.00,1.00,3.00,3.00,3.00,4.00,2.00,2.00,2.0,2.00,2.00,3.00,1.00,5.00,9.25,2.75,2.67,2.67,3.00,2.67,45.00
50%,3.0,3.00,3.00,1.00,3.00,4.00,3.00,4.00,4.00,2.00,4.0,2.00,2.00,3.00,3.00,5.00,10.00,3.25,2.67,2.67,3.00,2.67,45.00
75%,3.0,3.00,3.00,2.00,4.00,4.00,3.00,4.00,4.00,2.00,4.0,2.00,2.00,4.00,4.00,5.00,11.00,3.25,2.67,3.00,3.00,3.00,46.00
max,3.0,4.00,4.00,2.00,4.00,6.00,4.00,5.00,4.00,7.00,6.0,3.00,2.00,6.00,6.00,6.00,12.00,4.25,3.33,3.33,3.00,3.33,55.00


# 4 Metrics

In [611]:
name = {'ID' : 'ID',
        'B11' :  'school_grade',
        'C3' : 'game(weekdays)',
        'C4' : 'game(weekend)',
        'F10' : 'occupation(father)',
        'F11' : 'occupation(mother)',
        'F7_1' : 'age(father)',
        'F7_2' : 'age(mother)',
        'F8' : 'education(mother)',
        'F9' : 'education(father)'
        }

name_multi = {'ID' : 'ID',
        'B11' :  'school_grade',
        'C3' : 'game(weekdays)',
        'C4' : 'game(weekend)',
        'F10' : 'occupation(father)',
        'F11' : 'occupation(mother)',
        'F7_1' : 'age(father)',
        'F7_2' : 'age(mother)',
        'F8' : 'education(mother)',
        'F9' : 'education(father)',
        'A3' : 'national_identity',
        'B2_16' : 'korean(parents)',
        'B10' : 'korean(oneself)',
        'B17' : 'discrimination'
        }

def rename(df, mapping):
    df_renamed = df.rename(columns=mapping)
    return df_renamed[list(mapping.values())]

In [612]:
synthetic1_llama_p = synthetic1_llama_p.reset_index()
synthetic2_llama_p = synthetic2_llama_p.reset_index()
synthetic1_gemmni_p = synthetic1_gemmni_p.reset_index()
synthetic2_gemmni_p = synthetic2_gemmni_p.reset_index()

In [613]:
synthetic1_llama_p = rename(synthetic1_llama_p, name)
synthetic2_llama_p = rename(synthetic2_llama_p, name_multi)
synthetic1_gemmni_p = rename(synthetic1_gemmni_p, name)
synthetic2_gemmni_p = rename(synthetic2_gemmni_p, name_multi)

In [614]:
sample1 = rename(sample1, name)
sample2 = rename(sample2, name_multi)

In [615]:
synthetic2_gemmni_p.head()

Number,ID,school_grade,game(weekdays),game(weekend),occupation(father),occupation(mother),age(father),age(mother),education(mother),education(father),national_identity,korean(parents),korean(oneself),discrimination
0,6901.0,3.0,3.0,4.0,2.0,4.0,2.0,2.0,3.0,1.0,3.0,4.0,3.0,1.0
1,31904.0,3.0,3.0,4.0,2.0,4.0,2.0,2.0,3.0,1.0,3.0,3.0,3.0,2.0
2,37001.0,3.0,3.0,4.0,2.0,2.0,2.0,2.0,3.0,4.0,3.0,3.0,3.0,1.0
3,55502.0,3.0,3.0,4.0,2.0,2.0,2.0,2.0,3.0,4.0,3.0,4.0,3.0,2.0
4,56402.0,3.0,2.0,3.0,7.0,4.0,3.0,2.0,6.0,1.0,3.0,2.0,3.0,1.0


In [616]:
synthetic1_llama_p.head()

Number,ID,school_grade,game(weekdays),game(weekend),occupation(father),occupation(mother),age(father),age(mother),education(mother),education(father)
0,3711.0,3.0,3.0,4.0,2.0,4.0,2.0,2.0,3.0,3.0
1,16006.0,3.0,3.0,4.0,2.0,4.0,2.0,2.0,3.0,3.0
2,17011.0,3.0,3.0,3.0,2.0,4.0,2.0,2.0,3.0,3.0
3,21608.0,1.0,3.0,3.0,2.0,4.0,2.0,2.0,3.0,3.0
4,30405.0,3.0,3.0,3.0,3.0,4.0,2.0,2.0,3.0,3.0


In [617]:
%pip install pingouin
%pip install pyrelimri


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [618]:
from scipy.stats import chi2_contingency
from sklearn.metrics import cohen_kappa_score
from pyrelimri.tetrachoric_correlation import tetrachoric_corr
import pingouin as pg

In [619]:
# 공통 컬럼 추출
common_cols = list(set(sample1.columns) & set(synthetic1_llama_p.columns))

# 공통 인덱스 추출
common_idx = sample1.index.intersection(synthetic1_llama_p.index)

# 데이터 정렬
sample1_aligned = sample1.loc[common_idx, common_cols]
synthetic1_llama_p_aligned = synthetic1_llama_p.loc[common_idx, common_cols]

In [620]:
# 공통 컬럼 추출
common_cols = list(set(sample1.columns) & set(synthetic1_gemmni_p.columns))

# 공통 인덱스 추출
common_idx = sample1.index.intersection(synthetic1_gemmni_p.index)

# 데이터 정렬
sample1_aligned = sample1.loc[common_idx, common_cols]
synthetic1_gemmni_p_aligned = synthetic1_gemmni_p.loc[common_idx, common_cols]

In [621]:
# 공통 컬럼 추출
common_cols = list(set(sample2.columns) & set(synthetic2_llama_p.columns))

# 공통 인덱스 추출
common_idx = sample2.index.intersection(synthetic2_llama_p.index)

# 데이터 정렬
sample2_aligned = sample2.loc[common_idx, common_cols]
synthetic2_llama_p_aligned = synthetic2_llama_p.loc[common_idx, common_cols]

In [622]:
# 공통 컬럼 추출
common_cols = list(set(sample2.columns) & set(synthetic2_gemmni_p.columns))

# 공통 인덱스 추출
common_idx = sample2.index.intersection(synthetic2_gemmni_p.index)

# 데이터 정렬
sample2_aligned = sample2.loc[common_idx, common_cols]
synthetic2_gemmni_p_aligned = synthetic2_gemmni_p.loc[common_idx, common_cols]

In [623]:
def calculate_metrics_per_variable(real_series, synth_series):
    """변수별 지표 계산 함수"""
    # 결측치 제거 및 공통 인덱스 추출
    common_idx = real_series.dropna().index.intersection(
                synth_series.dropna().index)
    
    if len(common_idx) < 10:  # 최소 10개 샘플
        return None
    
    real_vals = real_series.loc[common_idx]
    synth_vals = synth_series.loc[common_idx]
    
    # 결과 저장 딕셔너리
    results = {
        'variable': real_series.name,
        'n_samples': len(common_idx),
        'n_unique_real': real_vals.nunique(),
        'n_unique_synth': synth_vals.nunique()
    }
    
    # 1. Intraclass Correlation (ICC)
    try:
        # ICC 계산을 위한 long-format 데이터 생성
        df_long = pd.DataFrame({
            'Subject': np.concatenate([common_idx, common_idx]),
            'Rater': ['Real']*len(common_idx) + ['Synth']*len(common_idx),
            'Rating': np.concatenate([real_vals.values, synth_vals.values])
        })
        
        icc = pg.intraclass_corr(
            data=df_long,
            targets='Subject',
            raters='Rater',
            ratings='Rating',
            nan_policy='omit'
        )
        # ICC3 (Single fixed raters) 추출
        icc3 = icc[icc['Type'] == 'ICC3']['ICC'].values[0]
        results['icc'] = icc3
    except:
        results['icc'] = np.nan
    
    # 2. Cohen's Kappa
    try:
        results['kappa'] = cohen_kappa_score(real_vals, synth_vals)
    except:
        results['kappa'] = np.nan
    
    # 3. Proportion Agreement
    results['agreement'] = (real_vals == synth_vals).mean()
    
    return results

def calculate_all_metrics(df_real, df_synth):
    """전체 데이터셋에 대해 변수별 지표 계산"""
    metrics_list = []
    
    for col in df_real.columns:
        if col not in df_synth.columns:
            continue
            
        metrics = calculate_metrics_per_variable(
            df_real[col], 
            df_synth[col]
        )
        
        if metrics:
            metrics_list.append(metrics)
    
    return pd.DataFrame(metrics_list)


In [624]:
def calculate_all_metrics(df_real, df_synth):
    """전체 데이터셋에 대해 변수별 지표 계산"""
    metrics_list = []
    
    for col in df_real.columns:
        if col not in df_synth.columns:
            continue
            
        metrics = calculate_metrics_per_variable(
            df_real[col], 
            df_synth[col]
        )
        
        if metrics:
            metrics_list.append(metrics)
    
    return pd.DataFrame(metrics_list)

In [625]:
# 변수별 지표 계산
metrics_llama = calculate_all_metrics(sample1_aligned, synthetic1_llama_p_aligned)
metrics_gemini = calculate_all_metrics(sample1_aligned, synthetic1_gemmni_p_aligned)

# 결과 출력
print("Llama 결과 (변수별):")
print(metrics_llama[['variable', 'icc', 'kappa', 'agreement']].round(2))

print("\nGemini 결과 (변수별):")
print(metrics_gemini[['variable', 'icc', 'kappa', 'agreement']].round(2))

Llama 결과 (변수별):
             variable   icc  kappa  agreement
0                  ID  0.13   0.02       0.04
1         age(father) -0.03  -0.04       0.75
2  occupation(father) -0.17  -0.08       0.28
3   education(father)  0.10   0.00       0.45
4  occupation(mother) -0.17  -0.01       0.14
5       game(weekend)  0.00  -0.05       0.22
6   education(mother) -0.04   0.00       0.32
7         age(mother)  0.25   0.22       0.80
8      game(weekdays) -0.09   0.00       0.16
9        school_grade  0.20   0.12       0.40

Gemini 결과 (변수별):
             variable   icc  kappa  agreement
0                  ID  0.13   0.02       0.04
1         age(father)  0.13   0.07       0.75
2  occupation(father) -0.19  -0.10       0.30
3   education(father) -0.03   0.11       0.49
4  occupation(mother)  0.06  -0.04       0.22
5       game(weekend)  0.13   0.06       0.26
6   education(mother)  0.02   0.13       0.43
7         age(mother)  0.29   0.22       0.76
8      game(weekdays)  0.06  -0.06       0.10


In [626]:
def align_dataframes(df_real, df_synth):
    """공통 컬럼만 남기고 정렬하는 함수"""
    # 공통 컬럼 추출
    common_cols = list(set(df_real.columns) & set(df_synth.columns))
    
    # 공통 인덱스 추출
    common_idx = df_real.index.intersection(df_synth.index)
    
    # 데이터 정렬
    df_real_aligned = df_real.loc[common_idx, common_cols]
    df_synth_aligned = df_synth.loc[common_idx, common_cols]
    
    return df_real_aligned, df_synth_aligned

In [627]:
# 변수별 지표 계산
metrics2_llama = calculate_all_metrics(sample2_aligned, synthetic2_llama_p_aligned)
metrics2_gemini = calculate_all_metrics(sample2_aligned, synthetic2_gemmni_p_aligned)
    
# 결과 출력
print("Llama 결과 (변수별):")
print(metrics2_llama[['variable', 'icc', 'kappa', 'agreement']].round(2))

print("\nGemini 결과 (변수별):")
print(metrics2_gemini[['variable', 'icc', 'kappa', 'agreement']].round(2))

Llama 결과 (변수별):
              variable   icc  kappa  agreement
0                   ID  0.18   0.02       0.04
1          age(father) -0.17   0.05       0.53
2         school_grade  0.02  -0.16       0.28
3   occupation(father)  0.06   0.04       0.27
4    education(father)  0.06  -0.03       0.35
5   occupation(mother)  0.05  -0.02       0.10
6       discrimination  0.08  -0.01       0.24
7        game(weekend)  0.00  -0.08       0.16
8    education(mother) -0.03   0.14       0.53
9          age(mother) -0.24  -0.14       0.34
10     korean(parents)  0.00   0.00       0.32
11   national_identity  0.00   0.00       0.32
12      game(weekdays) -0.06  -0.02       0.24
13     korean(oneself)  0.04   0.01       0.18

Gemini 결과 (변수별):
              variable   icc  kappa  agreement
0                   ID  0.18   0.02       0.04
1          age(father)  0.02   0.06       0.55
2         school_grade -0.14  -0.20       0.26
3   occupation(father) -0.02  -0.02       0.22
4    education(father) -0.

In [628]:
metrics2_llama  = pd.DataFrame(metrics2_llama[['variable', 'icc', 'kappa', 'agreement']])
metrics2_gemini = pd.DataFrame(metrics2_gemini[['variable', 'icc', 'kappa', 'agreement']])

final_order = [
    'age(father)',          # Father's age
    'age(mother)',          # Mother's age
    'education(father)',    # Father's education level
    'education(mother)',    # Mother's education level
    'occupation(father)',   # Father's occupation
    'occupation(mother)',   # Mother's occupation
    'game(weekdays)',       # Average Game Time (평일 기준)
    'game(weekend)',       # Average Game Time (평일 기준)
    'national_identity',    # National Identity
    'korean(parents)',      # Parents' Korean Language Proficiency
    'korean(oneself)',
    'discrimination'       # Korean Language Proficiency
]

metrics2_llama  = pd.DataFrame(metrics2_llama[['variable', 'icc', 'kappa', 'agreement']])
metrics2_gemini = pd.DataFrame(metrics2_gemini[['variable', 'icc', 'kappa', 'agreement']])

metrics2_llama_ordered = metrics2_llama.set_index('variable').loc[final_order].reset_index()
metrics2_gemini_ordered = metrics2_gemini.set_index('variable').loc[final_order].reset_index()

In [629]:
metrics2_llama_ordered.round(2)

,variable,icc,kappa,agreement
0,age(father),-0.17,0.05,0.53
1,age(mother),-0.24,-0.14,0.34
2,education(father),0.06,-0.03,0.35
3,education(mother),-0.03,0.14,0.53
4,occupation(father),0.06,0.04,0.27
5,occupation(mother),0.05,-0.02,0.10
6,game(weekdays),-0.06,-0.02,0.24
7,game(weekend),0.00,-0.08,0.16
8,national_identity,0.00,0.00,0.32
9,korean(parents),0.00,0.00,0.32


In [630]:
metrics2_gemini_ordered.round(2)

,variable,icc,kappa,agreement
0,age(father),0.02,0.06,0.55
1,age(mother),-0.00,0.00,0.57
2,education(father),-0.05,0.01,0.21
3,education(mother),-0.12,0.28,0.55
4,occupation(father),-0.02,-0.02,0.22
5,occupation(mother),-0.10,-0.09,0.14
6,game(weekdays),0.10,0.07,0.30
7,game(weekend),0.05,0.04,0.20
8,national_identity,0.00,0.00,0.32
9,korean(parents),-0.00,-0.03,0.12
